<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/KoboSurveyDataDownload/DownloadData%26Media_from_KobotoolBox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. CONFIGURATION SETTINGS
# ==============================================================================

import os
import requests
import json
import pandas as pd
from tqdm.notebook import tqdm  # Use tqdm.notebook for clean progress bars in Google Colab

# KoboToolbox Server URL (Use 'https://kf.kobotoolbox.org' or 'https://kobo.humanitarianresponse.info')
KOBO_URL = "https://kf.kobotoolbox.org"
# Your KoboToolbox API Token (Find in Account Settings > API Tokens)
API_TOKEN = "101508eb61f5625546c866c3dfbb55b5fec8fe67"
# Project Asset UID (Find in the URL: .../forms/aXYZ123456789/...)
ASSET_UID = "arDwQ2JShtAZzV9GtrfneU"


MEDIA_DIR = "/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_media_files"
OUTPUT_EXCEL = "/content/drive/MyDrive/Kobo-SpecLibFieldData/kobo_survey_data_local_media.xlsx"

HEADERS = {"Authorization": f"Token {API_TOKEN}"}
os.makedirs(MEDIA_DIR, exist_ok=True)

# ==============================================================================
# 2. FETCH ALL SUBMISSION DATA (PAGINATED)
# ==============================================================================
print("Fetching submission data from KoboToolbox API...")
data_url = f"{KOBO_URL}/api/v2/assets/{ASSET_UID}/data.json?page_size=1000"

results = []
while data_url:
    response = requests.get(data_url, headers=HEADERS)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: {response.status_code} - {response.text}")

    data_json = response.json()
    batch = data_json.get("results", [])
    results.extend(batch)
    data_url = data_json.get("next")

print(f"Successfully retrieved all {len(results)} submission records!\n")

# ==============================================================================
# 3. DOWNLOAD MEDIA ATTACHMENTS WITH PROGRESS BAR
# ==============================================================================
# Extract all valid media attachments first to set up accurate progress bar total
media_tasks = []
for record in results:
    attachments = record.get("_attachments", [])
    for attachment in attachments:
        download_url = attachment.get("download_small_url") or attachment.get("download_url")
        filename = attachment.get("filename", "").split("/")[-1]
        if download_url and filename:
            media_tasks.append((download_url, filename))

print(f"Starting download of {len(media_tasks)} total media attachments...")

# Progress bar initialization
for download_url, filename in tqdm(media_tasks, desc="Downloading Media Files", unit="file"):
    local_filepath = os.path.join(MEDIA_DIR, filename)

    # Download file if not present locally
    if not os.path.exists(local_filepath):
        img_res = requests.get(download_url, headers=HEADERS, stream=True)
        if img_res.status_code == 200:
            with open(local_filepath, "wb") as f:
                for chunk in img_res.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

# ==============================================================================
# 4. RELINK PATHS & EXPORT TO EXCEL
# ==============================================================================
processed_records = []
for record in results:
    record_dict = dict(record)
    attachments = record_dict.get("_attachments", [])
    for attachment in attachments:
        filename = attachment.get("filename", "").split("/")[-1]
        if filename:
            local_filepath = os.path.join(MEDIA_DIR, filename)
            for key, val in list(record_dict.items()):
                if isinstance(val, str) and filename in val:
                    record_dict[key] = local_filepath
    processed_records.append(record_dict)

df = pd.DataFrame(processed_records)
if "_attachments" in df.columns:
    df.drop(columns=["_attachments"], inplace=True)

df.to_excel(OUTPUT_EXCEL, index=False)
print(f"\nProcessing Complete!")
print(f"Saved {len(media_tasks)} media files to: '{os.path.abspath(MEDIA_DIR)}'")
print(f"Saved Excel dataset to: '{os.path.abspath(OUTPUT_EXCEL)}'")

Fetching submission data from KoboToolbox API...
Successfully retrieved all 632 submission records!

Starting download of 1892 total media attachments...
